# Product Review Sentimental Analysis using Naive Bayes

## 1. Importing libraries

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings(action='ignore')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

## 2. Loading dataset

In [2]:
data = pd.read_csv('Amazon_review_dataset.csv')
data.head()

,Review,Sentiment
0,Fast shipping but this product is very cheaply...,1
1,This case takes so long to ship and it's not e...,1
2,Good for not droids. Not good for iPhones. You...,1
3,The cable was not compatible between my macboo...,1
4,The case is nice but did not have a glow light...,1


In [3]:
data['Sentiment'].value_counts()

Sentiment
1    5000
2    5000
3    5000
4    5000
5    5000
Name: count, dtype: int64

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Review     24999 non-null  object
 1   Sentiment  25000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 390.8+ KB


## 3. Data preprocessing

In [6]:
# Removing the neutral sentiment
data = data[data['Sentiment'] != 3]
#  Converting the Sentimental rating (1,2,4 & 5) as sentiment <= 2 as 0(bad) and sentiment > 2 as 1(good).
data['Sentiment'] = np.where(data['Sentiment'] <= 2, 0, 1)

In [7]:
data.shape

(20000, 2)

In [8]:
data['Sentiment'].value_counts()

Sentiment
0    10000
1    10000
Name: count, dtype: int64

In [9]:
data.head()

,Review,Sentiment
0,Fast shipping but this product is very cheaply...,0
1,This case takes so long to ship and it's not e...,0
2,Good for not droids. Not good for iPhones. You...,0
3,The cable was not compatible between my macboo...,0
4,The case is nice but did not have a glow light...,0


In [11]:
data['Length'] = data['Review'].astype(str).apply(len)

In [12]:
data.head()

,Review,Sentiment,Length
0,Fast shipping but this product is very cheaply...,0,227
1,This case takes so long to ship and it's not e...,0,71
2,Good for not droids. Not good for iPhones. You...,0,146
3,The cable was not compatible between my macboo...,0,122
4,The case is nice but did not have a glow light...,0,112


In [13]:
data['Length'].describe()

count    20000.00000
mean       358.17510
std        513.74014
min          1.00000
25%        120.00000
50%        205.00000
75%        412.00000
max      15829.00000
Name: Length, dtype: float64

In [14]:
data[data['Length'] <= 1]['Review']

1161     😷
15049    a
15162    1
15978    👍
16929    👍
24799    a
Name: Review, dtype: object

In [15]:
# Emojis, number, and symbols usually don’t help Naive Bayes much so it's better to remove.
import re
import string
def clean_text(text):
    text = str(text) #Covert to string
    text = text.lower() #Lowercase
    text = "".join([i for i in text if i not in string.punctuation]) # Remove punctuations
    text = re.sub(r'[^a-zA-Z\s]', '', text) # Remove numbers, emojis and special characters
    text = re.sub(r'\s+', ' ', text).strip() # Remove extra spaces
    return text

data['Clean_review'] = data['Review'].apply(clean_text)

In [16]:
data.head()

,Review,Sentiment,Length,Clean_review
0,Fast shipping but this product is very cheaply...,0,227,fast shipping but this product is very cheaply...
1,This case takes so long to ship and it's not e...,0,71,this case takes so long to ship and its not ev...
2,Good for not droids. Not good for iPhones. You...,0,146,good for not droids not good for iphones you c...
3,The cable was not compatible between my macboo...,0,122,the cable was not compatible between my macboo...
4,The case is nice but did not have a glow light...,0,112,the case is nice but did not have a glow light...


In [17]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20000 entries, 0 to 24999
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Review        19999 non-null  object
 1   Sentiment     20000 non-null  int64 
 2   Length        20000 non-null  int64 
 3   Clean_review  20000 non-null  object
dtypes: int64(2), object(2)
memory usage: 781.2+ KB


## 4. TF-IDF Vectorization

In [19]:
tfidf = TfidfVectorizer(
    stop_words='english',
    max_df=0.7, # Removes words that appear in more than 70% of documents
    min_df=5, # Keeps only words that appear in at least 5 reviews
    ngram_range=(1,2)
)
X = tfidf.fit_transform(data['Clean_review'])

In [20]:
pd.DataFrame(X.toarray(), columns=list(tfidf.get_feature_names_out()))

,aa,aa batteries,aa battery,aaa,aaa batteries,aaa battery,aac,ab,ability,ability play,...,zone,zoom,zooming,zumo,zune,zune case,zune did,zune gb,zune hd,zunes
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19996,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.092576,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19998,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 5. Defining X and y

In [21]:
X = tfidf.fit_transform(data['Clean_review'])
y = data['Sentiment']

## 6. Splitting train and test data

In [22]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 7. Model Creation

In [24]:
model = MultinomialNB(alpha=0.1) #smoothing
model.fit(X_train, y_train)

,alpha,0.1
,force_alpha,True
,fit_prior,True
,class_prior,None


In [25]:
y_pred = model.predict(X_test)

## 8. Model evaluation

In [26]:
Accuracy = accuracy_score(y_test, y_pred)
Accuracy

0.8545

## Testing with new reviews

In [30]:
# Positive Review
review = "This product is really good and works perfectly"
clean_review = clean_text(review)
test = tfidf.transform([clean_review])
predict = model.predict(test)
print(predict)
if predict == 0:
    print('Negative review!')
else:
    print('Positive review!')

[1]
Positive review!


In [31]:
# Negative Review
review = "Worst experience, the product stopped working."
clean_review = clean_text(review)
test = tfidf.transform([clean_review])
predict = model.predict(test)
print(predict)
if predict == 0:
    print('Negative review!')
else:
    print('Positive review!')

[0]
Negative review!


In [33]:
import pickle
pickle.dump(model, open('model.pkl', 'wb'))
pickle.dump(tfidf, open('tfidf.pkl', 'wb'))